# 79장 · JavaScript 폼 이벤트와 fetch 응답 처리

웹 폼은 제출 이벤트에서 입력값을 읽어 요청 본문을 만듭니다. 비동기 응답을 받으면 상태를 확인한 뒤 화면에 표시할 값을 꺼냅니다.

과정 안내 화면의 submit 이벤트에서 요청 JSON을 만들고, HTTP 응답을 결과 영역에 반영합니다. 노트북에서는 이벤트와 응답을 재현해 정상·오류 경로를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## submit 이벤트와 JSON 요청 본문

브라우저의 submit 처리기는 event.preventDefault()로 기본 페이지 이동을 막고 입력값의 양끝을 정리합니다. 노트북에서는 Node.js가 설치된 수업 커널에서 실제 JavaScript를 실행합니다. DOM을 띄우지 않고 표준 EventTarget을 이용해 제출 동작만 재현합니다.


### 제출 이벤트가 읽는 입력값

<p>submit 이벤트에서 preventDefault()로 기본 폼 전송을 멈추고 입력값의 양끝 공백을 trim()으로 제거합니다. JSON.stringify가 만든 문자열을 보면 question은 문자열이고 k는 숫자입니다.</p>


### 셀 01 · JavaScript 실행 환경 준비

코드를 바꿀 곳은 다음 셀의 JavaScript입니다. 제공 함수는 실행 결과를 노트북에 표시합니다.


In [1]:
import subprocess
def run_js(source):
    run = subprocess.run(["node", "--input-type=module", "-e", source],
                         text=True, capture_output=True, check=True, timeout=10)
    print(run.stdout.rstrip())

함수를 정의했습니다. 별도의 JS 파일을 만들 필요가 없습니다.


### 셀 02 · submit 이벤트의 입력값

질문 양끝에 있는 공백이 JSON 본문에 남을지 예측하세요.


In [2]:
run_js("""
const form = new EventTarget();
const input = {value: "  과제는 어디에 내나요?  "};
form.addEventListener("submit", event => {
  event.preventDefault();
  const body = {question: input.value.trim(), k: 2};
  console.log(JSON.stringify(body));
});
form.dispatchEvent(new Event("submit", {cancelable: true}));
""")

{"question":"과제는 어디에 내나요?","k":2}


출력된 JSON의 question은 양끝 공백을 뺀 문자열이고 k는 숫자 2입니다. 이 셀은 요청 본문을 만드는 단계에서 끝납니다. 다음 셀에서 응답을 만들고 표시할 답변을 읽습니다.


### 대화 경로와 질문 본문의 역할

<p>예제의 경로에는 chat-a가, JSON 본문에는 question과 k가 들어갑니다. 경로는 어느 대화로 보낼지를, 본문은 이번에 무엇을 물을지를 나타냅니다. 이 셀은 네트워크에 접속하지 않고 fixtureFetch로 응답을 만듭니다.</p>

<h3>chat-a의 모의 응답에서 읽는 값</h3>

<p>셀 03은 대화가 이미 있다는 가정 아래 /conversations/chat-a/ask를 사용합니다. fixtureFetch가 질문을 읽고 result.answer를 돌려주면 예제 객체의 textContent에 넣습니다. 대화 생성이나 DB 저장을 실행하는 셀은 없습니다.</p>


## 비동기 응답과 오류 분기

실제 화면의 transport.mjs는 fetch 함수를 전달받습니다. 이 셀에서는 수업용 응답을 주는 함수를 넣어 요청과 화면 사이를 재현합니다. 실제 서버 주소에서는 해당 함수 자리에 브라우저 fetch를 넣습니다.


### 응답 도착과 JSON 읽기의 두 await

<p>첫 await는 모의 함수가 돌려주는 Response를 기다립니다. 성공 상태를 확인한 뒤 두 번째 await로 JSON 본문을 읽습니다. 응답 객체 자체에는 아직 result.answer 형태의 속성이 없습니다.</p>


### 셀 03 · 응답 JSON을 모의 객체의 textContent에 대입

fetch 응답과 JSON 해석을 각각 await합니다. 여기서 element는 화면 요소의 textContent 속성만 흉내 낸 객체입니다.


In [3]:
run_js("""
const fixtureFetch = async (_url, options) => {
  const { question } = JSON.parse(options.body);
  if (question !== "과제는 어디에 내나요?") return new Response(null, {status: 422});
  return Response.json({result: {answer: "과제함에 제출하세요. [course-guide-v1:p2]"}});
};
const response = await fixtureFetch("/conversations/chat-a/ask", {
  method: "POST", headers: {"Content-Type":"application/json"},
  body: JSON.stringify({question:"과제는 어디에 내나요?", k:2})
});
if (!response.ok) throw Error("HTTP " + response.status);
const body = await response.json();
const element = {textContent: ""};
element.textContent = body.result.answer;
console.log(element.textContent);
""")

과제함에 제출하세요. [course-guide-v1:p2]


표시할 텍스트에는 출처 ID가 남았습니다. innerHTML로 문자열을 해석하지 않고 textContent에 넣으면 답에 꺾쇠가 있더라도 문구로 다룰 수 있습니다.


### result.answer에서 textContent까지

<p>답변 문자열은 응답 맨 위가 아니라 result 안에 있습니다. 예제는 이 문자열을 textContent에 대입한 뒤 출력합니다. 실제 DOM에서 textContent에 대입해도 문자열을 HTML로 해석하지 않습니다.</p>


### 과제 04 · HTTP 실패 시 분기

본문이 빈 404 응답을 받더라도 JSON 파싱 오류가 나지 않게 처리하세요. 앞 셀의 성공 처리와 같은 위치에서 상태 코드를 먼저 읽고 안내 문구를 출력하세요.


In [4]:
# Node의 Response(null, {status: 404})를 사용해 상태별 처리를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(&quot;&quot;&quot;
const response = new Response(null, {status: 404});
if (response.status === 404) {
  console.log(&quot;대화를 찾을 수 없습니다&quot;);
} else if (response.ok) {
  const body = await response.json();
  console.log(body.result.answer);
} else {
  console.log(&quot;HTTP &quot; + response.status);
}
&quot;&quot;&quot;)</code></pre>
<p>실행 결과</p><pre><code>대화를 찾을 수 없습니다</code></pre>
<p>빈 본문에 response.json()을 먼저 호출하면 예외가 나서 안내 문구까지 가지 못합니다. 여기서는 Response를 직접 만들었습니다. 실제 연결이 끊겨 응답 객체 자체를 받지 못한 경우는 별도의 예외 처리 대상입니다.</p>
</details>


### 404 응답을 받았을 때 표시할 문구

<p>404인 Response는 빈 본문을 가질 수 있습니다. 이 경우에는 상태별 안내를 먼저 선택해 JSON 파싱 오류가 실제 HTTP 상태를 가리지 않게 합니다.</p>


## 돌아보기

제출 이벤트의 입력에는 양끝 공백이 있고 JSON 요청에는 없습니다. 응답이 실패했을 때 입력을 다시 보내기 전에는 어느 대화 ID와 질문이 요청에 실렸는지 확인합니다.
